# twisted smoke test

Event-driven networking framework. Exercises core APIs: Deferred, reactor, internet/web protocols, defer.inlineCallbacks, and reflect.

In [ ]:
# Example 1: Deferred — the core async primitive in Twisted.
from twisted.internet.defer import Deferred, succeed, fail

results = []

d = Deferred()
d.addCallback(lambda x: x * 2)
d.addCallback(lambda x: x + 1)
d.addCallback(lambda x: results.append(x) or x)
d.callback(10)

# succeed/fail produce already-resolved Deferreds.
d2 = succeed("hello")
d2.addCallback(lambda s: results.append(s.upper()) or s.upper())

errors = []
d3 = fail(ValueError("boom"))
d3.addErrback(lambda failure: errors.append(failure.type.__name__) or None)

print(f"results = {results}")
print(f"errors = {errors}")
assert results == [21, "HELLO"]
assert errors == ["ValueError"]

In [ ]:
# Example 2: DeferredList — combine multiple Deferreds.
from twisted.internet.defer import Deferred, DeferredList, succeed

collected = []

d1 = succeed("a")
d2 = succeed("b")
d3 = succeed("c")

dl = DeferredList([d1, d2, d3], consumeErrors=True)
dl.addCallback(lambda results: collected.extend(r[1] for r in results if r[0]))

print(f"collected = {collected}")
assert collected == ["a", "b", "c"]

# gatherResults variant.
from twisted.internet.defer import gatherResults
gathered = []
g = gatherResults([succeed(i * i) for i in range(5)])
g.addCallback(lambda vals: gathered.extend(vals))
print(f"gathered = {gathered}")
assert gathered == [0, 1, 4, 9, 16]

In [ ]:
# Example 3: inlineCallbacks — generator-based async style.
from twisted.internet.defer import inlineCallbacks, succeed, returnValue

captured = []

@inlineCallbacks
def fetch_two_things():
    a = yield succeed("first")
    b = yield succeed("second")
    returnValue(f"{a}+{b}")

d = fetch_two_things()
d.addCallback(lambda val: captured.append(val))

print(f"captured = {captured}")
assert captured == ["first+second"]

In [ ]:
# Example 4: Protocol + transport — build a simple line protocol instance.
from twisted.protocols.basic import LineReceiver
from twisted.test.proto_helpers import StringTransport

class EchoLine(LineReceiver):
    delimiter = b"\n"
    received = []
    def lineReceived(self, line):
        self.received.append(line)
        self.sendLine(b"echo:" + line)

proto = EchoLine()
tr = StringTransport()
proto.makeConnection(tr)
proto.dataReceived(b"hello\nworld\n")

out = tr.value()
print(f"received = {EchoLine.received}")
print(f"transport output = {out!r}")
assert EchoLine.received == [b"hello", b"world"]
assert b"echo:hello" in out
assert b"echo:world" in out

In [ ]:
# Example 5: twisted.python.reflect + filepath — utility surface used across the framework.
from twisted.python import reflect, filepath
import tempfile
import os

# namedAny: import-by-string used by twistd plugins.
Deferred_cls = reflect.namedAny("twisted.internet.defer.Deferred")
from twisted.internet.defer import Deferred
assert Deferred_cls is Deferred

qual = reflect.qual(Deferred)
print(f"qual(Deferred) = {qual}")
assert qual == "twisted.internet.defer.Deferred"

# FilePath: a path abstraction used throughout twisted.
with tempfile.TemporaryDirectory() as td:
    fp = filepath.FilePath(td).child("hello.txt")
    fp.setContent(b"twisted-test")
    assert fp.exists()
    assert fp.getContent() == b"twisted-test"
    print(f"FilePath roundtrip OK at {fp.path}")

print("twisted smoke test passed")
assert True